<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/llm/Grouped-Query-Attention/grouped-query-attention-Question.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("grouped-query-attention", ...)`


# Implement Attention from Scratch
### 🧠 Problem Statement
Standard Multi-Head Attention (MHA) assigns a separate query, key, and value projection to each attention head. But that’s not always the most efficient approach. 

Enter **Grouped Query Attention (GQA)** — a clever mechanism where you use more query heads than key-value heads. This reduces compute/memory costs while still allowing for fine-grained query specialization.

Your task is to **implement GQA from scratch** and validate it against PyTorch’s `MultiheadAttention` under the special case where GQA behaves identically to MHA (i.e., when `num_query_heads == num_query_groups`).

---

### ✅ Requirements

1. **Define the GQA Mechanism**
   - Create a function `grouped_query_attention(q, k, v, num_query_groups, d_model, mask=None)`.
   - Project `q`, `k`, and `v` using linear layers:
     - Q projection → all query heads.
     - K/V projection → shared across grouped key/value heads.
   - Use `repeat_interleave()` to expand grouped K/V heads to match the number of Q heads.

2. **Compute Attention**
   - Apply scaled dot-product attention using `Q @ Kᵀ / sqrt(d_head)`.
   - Support optional masking.
   - Return output by concatenating heads and applying the output projection.

3. **Validate Against MHA**
   - Test your implementation using synthetic tensors.
   - Compare your output to `torch.nn.MultiheadAttention` where GQA degenerates to MHA (`num_query_heads == num_query_groups`).
   - Assert that both outputs match numerically.

---

### 📏 Constraints

- ✅ Use only PyTorch (no external libraries like xformers or HuggingFace).
- ✅ Output shape must be `(batch_size, seq_len, d_model)`.
- ✅ Support optional attention masking.
- ✅ Validate output against `torch.nn.MultiheadAttention` for correctness.

---

<details>
  <summary>💡 Hint</summary>

  - Use `nn.Linear(d_model, d_model)` for projecting `q`, `k`, and `v`.
  - When `num_query_heads > num_query_groups`, use `.repeat_interleave()` to duplicate each group’s `K`/`V` to match query head count.
  - Final output: reshape the multi-head outputs to `(batch_size, seq_len, d_model)` and apply the output projection layer.
  - Test with `num_query_heads == num_query_groups` to confirm it behaves like MHA.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [ ]:
# Synthetic data
torch.manual_seed(42)
batch_size = 3
seq_len = 4
d_model = 8
num_heads = 2

q = torch.rand(batch_size, seq_len, d_model)
k = torch.rand(batch_size, seq_len, d_model)
v = torch.rand(batch_size, seq_len, d_model)
print(q.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def grouped_query_attention(q, k, v, num_query_heads, num_query_groups, d_model,
                            mask=None):
    """
    Implements Grouped Query Attention (GQA).

    Queries keep `num_query_heads` heads, but keys and values only have
    `num_query_groups` heads; each K/V head is shared by
    `num_query_heads // num_query_groups` query heads. With
    num_query_groups == num_query_heads this is ordinary multi-head attention,
    and with num_query_groups == 1 it is multi-query attention.

    Args:
        q, k, v (Tensor): (batch_size, seq_len, d_model)
        num_query_heads (int): number of query heads
        num_query_groups (int): number of key/value heads; must divide num_query_heads
        d_model (int): total embedding dimension
        mask (Tensor, optional): broadcastable to (batch, heads, seq, seq);
            positions equal to 0 are not attended to

    Returns:
        Tensor: (batch_size, seq_len, d_model)
    """
    ...



In [ ]:
# GQA with as many K/V groups as query heads is exactly MHA, so it should match
# torch.nn.MultiheadAttention up to the (random) projection weights - we compare
# shapes here and check the grouping behaviour separately.
num_query_heads = 4
d_model = 8

q = torch.rand(batch_size, seq_len, d_model)
k = torch.rand(batch_size, seq_len, d_model)
v = torch.rand(batch_size, seq_len, d_model)

for num_query_groups in (1, 2, 4):
    out = grouped_query_attention(
        q, k, v,
        num_query_heads=num_query_heads,
        num_query_groups=num_query_groups,
        d_model=d_model,
    )
    assert out.shape == (batch_size, seq_len, d_model), out.shape
    print(f"num_query_groups={num_query_groups}: {tuple(out.shape)}")

# A causal mask must stop position 0 from seeing later tokens.
causal = torch.tril(torch.ones(seq_len, seq_len)).bool()
masked = grouped_query_attention(
    q, k, v, num_query_heads=num_query_heads, num_query_groups=2,
    d_model=d_model, mask=causal,
)
print("masked output:", tuple(masked.shape))

